# Trees, forests & boosting (ISLP ch. 8): practice notebook

3 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("k1")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import statsmodels.api as sm

def rdata(name, package):
    """Load an R dataset (via statsmodels / Rdatasets) as a polars DataFrame, NaN → null."""
    df = pl.from_pandas(sm.datasets.get_rdataset(name, package, cache=True).data.reset_index(drop=True)).drop("rownames", strict=False)
    return df.with_columns(pl.col(pl.Float64).fill_nan(None))

carseats = rdata("Carseats", "ISLR")
oj = rdata("OJ", "ISLR")
hitters = rdata("Hitters", "ISLR")

def design(df, target):
    """Numeric design matrix: string columns one-hot encoded (first level dropped). Returns X, y, names."""
    cats = [c for c, t in df.schema.items() if t == pl.String and c != target]
    X = df.drop(target).to_dummies(cats, drop_first=True).cast(pl.Float64)
    return X.to_numpy(), df[target], X.columns

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJrMSI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdFxuZnJvbSBza2xlYXJuLnRyZWUgaW1wb3J0IERlY2lzaW9uVHJlZVJlZ3Jlc3NvclxuZnJvbSBza2xlYXJuLmVuc2VtYmxlIGltcG9ydCBSYW5kb21Gb3Jlc3RSZWdyZXNzb3JcbmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBtZWFuX3NxdWFyZWRfZXJyb3JcblxuZGVmIGNhcnNlYXRzX21vZGVscyhkZiwgc2VlZD0wKTpcbiAgICBYLCB5LCBuYW1lcyA9IGRlc2lnbihkZiwgXCJTYWxlc1wiKVxuICAgIFh0ciwgWHRlLCB5dHIsIHl0ZSA9IHRyYWluX3Rlc3Rfc3BsaXQoWCwgeS50b19udW1weSgpLCB0ZXN0X3NpemU9MC41LCByYW5kb21fc3RhdGU9c2VlZClcbiAgICBtb2RlbHMgPSB7XCJ0cmVlXCI6IERlY2lzaW9uVHJlZVJlZ3Jlc3NvcihyYW5kb21fc3RhdGU9MCksXG4gICAgICAgICAgICAgIFwiYmFnZ2luZ1wiOiBSYW5kb21Gb3Jlc3RSZWdyZXNzb3Iobl9lc3RpbWF0b3JzPTMwMCwgbWF4X2ZlYXR1cmVzPU5vbmUsIHJhbmRvbV9zdGF0ZT0wKSxcbiAgICAgICAgICAgICAgXCJmb3Jlc3RcIjogUmFuZG9tRm9yZXN0UmVncmVzc29yKG5fZXN0aW1hdG9ycz0zMDAsIG1heF9mZWF0dXJlcz0xLzMsIHJhbmRvbV9zdGF0ZT0wKX1cbiAgICBvdXQgPSB7azogZmxvYXQobWVhbl9zcXVhcmVkX2Vycm9yKHl0ZSwgbS5maXQoWHRyLCB5dHIpLnByZWRpY3QoWHRlKSkpIGZvciBrLCBtIGluIG1vZGVscy5pdGVtcygpfVxuICAgIGltcCA9IG1vZGVsc1tcImZvcmVzdFwiXS5mZWF0dXJlX2ltcG9ydGFuY2VzX1xuICAgIG91dFtcInRvcFwiXSA9IFtuYW1lc1tpXSBmb3IgaSBpbiBucC5hcmdzb3J0KGltcClbOjotMV1bOjJdXVxuICAgIHJldHVybiBvdXRcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3Qoc2VlZCk6XG4gICAgYSA9IFVbXCJjYXJzZWF0c19tb2RlbHNcIl0oY2Fyc2VhdHMsIHNlZWQpOyBiID0gUltcImNhcnNlYXRzX21vZGVsc1wiXShjYXJzZWF0cywgc2VlZCk7IHNhbWVfa2V5cyhhLCBiKVxuICAgIGZvciBrIGluIChcInRyZWVcIiwgXCJiYWdnaW5nXCIsIFwiZm9yZXN0XCIpOiBjbG9zZShhW2tdLCBiW2tdLCAwLjA4LCBrICsgXCIgdGVzdCBNU0VcIilcbiAgICBhc3NlcnQgYVtcImJhZ2dpbmdcIl0gPCBhW1widHJlZVwiXSBhbmQgYVtcImZvcmVzdFwiXSA8IGFbXCJ0cmVlXCJdLCBcInRoZSBlbnNlbWJsZXMgc2hvdWxkIGJlYXQgdGhlIHNpbmdsZSB0cmVlXCJcbiAgICBwcmUgPSBsYW1iZGEgdjoge3Muc3BsaXQoXCJfXCIpWzBdIGZvciBzIGluIHZ9XG4gICAgYXNzZXJ0IHByZShhW1widG9wXCJdKSA9PSBwcmUoYltcInRvcFwiXSksIGZcInRvcCBmZWF0dXJlcyB7YVsndG9wJ119LCBleHBlY3RlZCB7YlsndG9wJ119XCJcblRFU1RTID0gWyhcInNlZWQgMFwiLCBsYW1iZGE6IF90KDApKSwgKFwiaGlkZGVuOiBzZWVkIDdcIiwgbGFtYmRhOiBfdCg3KSldXG4iLCAibGFuZyI6ICJweSJ9LCAiazIiOiB7InNvbHV0aW9uIjogImZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQsIEdyaWRTZWFyY2hDVlxuZnJvbSBza2xlYXJuLnRyZWUgaW1wb3J0IERlY2lzaW9uVHJlZUNsYXNzaWZpZXJcblxuZGVmIG9qX3RyZWUoZGYsIHNlZWQ9MCk6XG4gICAgWCwgeSwgbmFtZXMgPSBkZXNpZ24oZGYsIFwiUHVyY2hhc2VcIilcbiAgICBYdHIsIFh0ZSwgeXRyLCB5dGUgPSB0cmFpbl90ZXN0X3NwbGl0KFgsIHkudG9fbnVtcHkoKSwgdHJhaW5fc2l6ZT04MDAsIHJhbmRvbV9zdGF0ZT1zZWVkKVxuICAgIGZ1bGwgPSBEZWNpc2lvblRyZWVDbGFzc2lmaWVyKHJhbmRvbV9zdGF0ZT0wKS5maXQoWHRyLCB5dHIpXG4gICAgYWxwaGFzID0gZnVsbC5jb3N0X2NvbXBsZXhpdHlfcHJ1bmluZ19wYXRoKFh0ciwgeXRyKS5jY3BfYWxwaGFzXG4gICAgZ3MgPSBHcmlkU2VhcmNoQ1YoRGVjaXNpb25UcmVlQ2xhc3NpZmllcihyYW5kb21fc3RhdGU9MCksIHtcImNjcF9hbHBoYVwiOiBhbHBoYXN9LCBjdj01KS5maXQoWHRyLCB5dHIpXG4gICAgcHJ1bmVkID0gZ3MuYmVzdF9lc3RpbWF0b3JfXG4gICAgZXJyID0gbGFtYmRhIG06IGZsb2F0KG5wLm1lYW4obS5wcmVkaWN0KFh0ZSkgIT0geXRlKSlcbiAgICByZXR1cm4ge1wiZnVsbF9lcnJvclwiOiBlcnIoZnVsbCksIFwiZnVsbF9sZWF2ZXNcIjogaW50KGZ1bGwuZ2V0X25fbGVhdmVzKCkpLFxuICAgICAgICAgICAgXCJwcnVuZWRfZXJyb3JcIjogZXJyKHBydW5lZCksIFwicHJ1bmVkX2xlYXZlc1wiOiBpbnQocHJ1bmVkLmdldF9uX2xlYXZlcygpKX1cbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3Qoc2VlZCk6XG4gICAgYSA9IFVbXCJval90cmVlXCJdKG9qLCBzZWVkKTsgYiA9IFJbXCJval90cmVlXCJdKG9qLCBzZWVkKTsgc2FtZV9rZXlzKGEsIGIpXG4gICAgYXNzZXJ0IGFbXCJwcnVuZWRfbGVhdmVzXCJdIDwgYVtcImZ1bGxfbGVhdmVzXCJdLCBcInBydW5pbmcgc2hvdWxkIHJlbW92ZSBsZWF2ZXNcIlxuICAgIGJldHdlZW4oYVtcInBydW5lZF9lcnJvclwiXSwgMC4wOCwgMC4zLCBcInBydW5lZCB0ZXN0IGVycm9yXCIpXG4gICAgY2xvc2UoYVtcImZ1bGxfZXJyb3JcIl0sIGJbXCJmdWxsX2Vycm9yXCJdLCAwLjE1LCBcImZ1bGwtdHJlZSB0ZXN0IGVycm9yXCIpXG5URVNUUyA9IFsoXCJzZWVkIDBcIiwgbGFtYmRhOiBfdCgwKSksIChcImhpZGRlbjogc2VlZCAzXCIsIGxhbWJkYTogX3QoMykpXVxuIiwgImxhbmciOiAicHkifSwgImszIjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4uZW5zZW1ibGUgaW1wb3J0IEdyYWRpZW50Qm9vc3RpbmdSZWdyZXNzb3JcbmZyb20gc2tsZWFybi5saW5lYXJfbW9kZWwgaW1wb3J0IExpbmVhclJlZ3Jlc3Npb25cblxuZGVmIGhpdHRlcnNfYm9vc3QoZGYsIHJhdGVzKTpcbiAgICBkID0gZGYuZHJvcF9udWxscyhcIlNhbGFyeVwiKS53aXRoX2NvbHVtbnMocGwuY29sKFwiU2FsYXJ5XCIpLmxvZygpKVxuICAgIFgsIHksIG5hbWVzID0gZGVzaWduKGQsIFwiU2FsYXJ5XCIpXG4gICAgeSA9IHkudG9fbnVtcHkoKVxuICAgIFh0ciwgWHRlLCB5dHIsIHl0ZSA9IFhbOjIwMF0sIFhbMjAwOl0sIHlbOjIwMF0sIHlbMjAwOl1cbiAgICBtc2UgPSBsYW1iZGEgbTogZmxvYXQobnAubWVhbigobS5maXQoWHRyLCB5dHIpLnByZWRpY3QoWHRlKSAtIHl0ZSkgKiogMikpXG4gICAgcmV0dXJuIHtcImJvb3N0XCI6IG5wLmFycmF5KFttc2UoR3JhZGllbnRCb29zdGluZ1JlZ3Jlc3NvcihuX2VzdGltYXRvcnM9MTAwMCwgbGVhcm5pbmdfcmF0ZT1yLCByYW5kb21fc3RhdGU9MCkpIGZvciByIGluIHJhdGVzXSksXG4gICAgICAgICAgICBcImxpbmVhclwiOiBtc2UoTGluZWFyUmVncmVzc2lvbigpKX1cbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoKTpcbiAgICByYXRlcyA9IFswLjAwNSwgMC4wNSwgMC4zXVxuICAgIGEgPSBVW1wiaGl0dGVyc19ib29zdFwiXShoaXR0ZXJzLCByYXRlcyk7IGIgPSBSW1wiaGl0dGVyc19ib29zdFwiXShoaXR0ZXJzLCByYXRlcyk7IHNhbWVfa2V5cyhhLCBiKVxuICAgIGNsb3NlKGFbXCJsaW5lYXJcIl0sIGJbXCJsaW5lYXJcIl0sIDFlLTYsIFwibGluZWFyIHRlc3QgTVNFXCIpXG4gICAgZm9yIGksIHIgaW4gZW51bWVyYXRlKHJhdGVzKTogY2xvc2UoYVtcImJvb3N0XCJdW2ldLCBiW1wiYm9vc3RcIl1baV0sIDAuMDUsIGZcImJvb3N0aW5nIHRlc3QgTVNFIGF0IHJhdGUge3J9XCIpXG4gICAgYXNzZXJ0IG1pbihhW1wiYm9vc3RcIl0pIDwgYVtcImxpbmVhclwiXSwgXCJib29zdGluZyBzaG91bGQgYmVhdCB0aGUgbGluZWFyIG1vZGVsXCJcblRFU1RTID0gWyhcImJvb3N0aW5nIHZzIGxpbmVhclwiLCBfdCldXG4iLCAibGFuZyI6ICJweSJ9fQ==").decode())
_SETUP = _b64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwb2xhcnMgYXMgcGwKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAppbXBvcnQgc3RhdHNtb2RlbHMuYXBpIGFzIHNtCgpkZWYgcmRhdGEobmFtZSwgcGFja2FnZSk6CiAgICAiIiJMb2FkIGFuIFIgZGF0YXNldCAodmlhIHN0YXRzbW9kZWxzIC8gUmRhdGFzZXRzKSBhcyBhIHBvbGFycyBEYXRhRnJhbWUsIE5hTiDihpIgbnVsbC4iIiIKICAgIGRmID0gcGwuZnJvbV9wYW5kYXMoc20uZGF0YXNldHMuZ2V0X3JkYXRhc2V0KG5hbWUsIHBhY2thZ2UsIGNhY2hlPVRydWUpLmRhdGEucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSkuZHJvcCgicm93bmFtZXMiLCBzdHJpY3Q9RmFsc2UpCiAgICByZXR1cm4gZGYud2l0aF9jb2x1bW5zKHBsLmNvbChwbC5GbG9hdDY0KS5maWxsX25hbihOb25lKSkKCmNhcnNlYXRzID0gcmRhdGEoIkNhcnNlYXRzIiwgIklTTFIiKQpvaiA9IHJkYXRhKCJPSiIsICJJU0xSIikKaGl0dGVycyA9IHJkYXRhKCJIaXR0ZXJzIiwgIklTTFIiKQoKZGVmIGRlc2lnbihkZiwgdGFyZ2V0KToKICAgICIiIk51bWVyaWMgZGVzaWduIG1hdHJpeDogc3RyaW5nIGNvbHVtbnMgb25lLWhvdCBlbmNvZGVkIChmaXJzdCBsZXZlbCBkcm9wcGVkKS4gUmV0dXJucyBYLCB5LCBuYW1lcy4iIiIKICAgIGNhdHMgPSBbYyBmb3IgYywgdCBpbiBkZi5zY2hlbWEuaXRlbXMoKSBpZiB0ID09IHBsLlN0cmluZyBhbmQgYyAhPSB0YXJnZXRdCiAgICBYID0gZGYuZHJvcCh0YXJnZXQpLnRvX2R1bW1pZXMoY2F0cywgZHJvcF9maXJzdD1UcnVlKS5jYXN0KHBsLkZsb2F0NjQpCiAgICByZXR1cm4gWC50b19udW1weSgpLCBkZlt0YXJnZXRdLCBYLmNvbHVtbnM=").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. ISLP 8 ex. 8: Carseats regression tree, then bagging and a random forest
*Medium · id `k1`*

<p>ISLP chapter 8, applied exercise 8. Predict <code>Sales</code> from everything else in <code>carseats</code>, using the <code>design</code> helper from the setup.</p>
<p>Write <code>carseats_models(df, seed=0)</code>: split <code>train_test_split(X, y, test_size=0.5, random_state=seed)</code>, then fit and score (test MSE) a <code>DecisionTreeRegressor(random_state=0)</code>, bagging = <code>RandomForestRegressor(n_estimators=300, max_features=None, random_state=0)</code>, and a random forest with <code>max_features=1/3</code> (same n_estimators and random_state). Return <code>{"tree": mse, "bagging": mse, "forest": mse, "top": [two most important features of the forest]}</code>.</p>

**Theory to know cold**
<ul><li>One deep tree has low bias, high variance. Bagging averages trees fitted on bootstrap samples, cutting variance. A random forest also samples features at each split, which decorrelates the trees and cuts variance further.</li><li>Out-of-bag error is a free estimate of test error for bagged models.</li></ul>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

def carseats_models(df: pl.DataFrame, seed=0) -> dict:
    X, y, names = design(df, "Sales")
    # your code here
    pass

# Sample run
try:
    print(carseats_models(carseats))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("k1")

---
## 2. ISLP 8 ex. 9: OJ classification tree, pruned by cross-validation
*Medium · id `k2`*

<p>ISLP chapter 8, applied exercise 9. Predict <code>Purchase</code> (CH or MM) in <code>oj</code>.</p>
<p>Write <code>oj_tree(df, seed=0)</code>: <code>train_test_split(X, y, train_size=800, random_state=seed)</code>; fit a full <code>DecisionTreeClassifier(random_state=0)</code>; then choose <code>ccp_alpha</code> from <code>cost_complexity_pruning_path</code> on the training set by 5-fold <code>GridSearchCV</code> (accuracy) and refit. Return <code>{"full_error", "full_leaves", "pruned_error", "pruned_leaves"}</code> (test error rates and <code>get_n_leaves()</code>).</p>

**Theory to know cold**
<ul><li>Cost-complexity pruning minimises error + α·|leaves|; each α gives a nested subtree, and CV picks α.</li><li>A pruned tree is far easier to explain and usually generalises at least as well.</li></ul>

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

def oj_tree(df: pl.DataFrame, seed=0) -> dict:
    X, y, names = design(df, "Purchase")
    # your code here
    pass

# Sample run
try:
    print(oj_tree(oj))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("k2")

---
## 3. ISLP 8 ex. 10: boosting on Hitters, learning rate vs test MSE
*Hard · id `k3`*

<p>ISLP chapter 8, applied exercise 10. Drop players with a null <code>Salary</code>, predict log(Salary); the first 200 rows are the training set, the rest the test set.</p>
<p>Write <code>hitters_boost(df, rates)</code>: for each learning rate fit <code>GradientBoostingRegressor(n_estimators=1000, learning_rate=rate, random_state=0)</code> and record the test MSE; also fit a <code>LinearRegression</code>. Return <code>{"boost": array of MSEs, "linear": mse}</code>. Plot MSE against the rate.</p>

**Theory to know cold**
<ul><li>Boosting fits trees to the residuals one after another; a small learning rate needs more trees but generalises better.</li><li>Unlike bagging, boosting reduces bias, and too many trees can overfit.</li></ul>

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

def hitters_boost(df: pl.DataFrame, rates) -> dict:
    d = df.drop_nulls("Salary").with_columns(pl.col("Salary").log())
    X, y, names = design(d, "Salary")
    # your code here
    pass

# Sample run
try:
    rates = [0.001, 0.005, 0.01, 0.05, 0.1, 0.3]
    res = hitters_boost(hitters, rates); print(res)
    plt.plot(rates, res["boost"], "o-"); plt.axhline(res["linear"], ls="--"); plt.xscale("log"); plt.show()
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("k3")